# CPU, cuDF, and Triton TPC-H-derived comparison

This notebook runs and visualizes the repository's PDS-H benchmark suite. PDS-H is derived from TPC-H and is not an official TPC-H publication benchmark. Configure the dataset and benchmark options in `PARAMETERS`, run the simulation cell, and then run the plotting cell. Each duration is appended to the analysis results folder at `q{query}/{scale}/{backend}[_exclude_conversion_time]`, and the plotting cell averages repeated durations.

In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess

import pandas as pd

repo_root = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
scale_factor = 5#float(os.environ.get("PDSH_SCALE_FACTOR", "1"))
scale_label = f"{scale_factor:g}"
triton_executor = "in-memory" if scale_factor <= 5 else "streaming"
results_dir = repo_root / "tpch_backend_results" / f"scale-{scale_label}"
analysis_results_dir = repo_root / "tpch_backend_results" / "analysis_results"
dataset_candidates = [
    Path(os.environ["PDSH_DATASET_PATH"])
    if os.environ.get("PDSH_DATASET_PATH")
    else None,
    repo_root.parent / "polars-benchmark" / "data" / "tables" / f"scale-{scale_label}.0",
    repo_root / "datasets" / "tables" / f"scale-{scale_label}.0",
]
dataset_candidates = [path for path in dataset_candidates if path is not None]
dataset_path = next(
    (path for path in dataset_candidates if (path / "lineitem.parquet").exists()),
    dataset_candidates[0]
    if dataset_candidates
    else repo_root / "datasets" / "tables" / f"scale-{scale_label}",
)

PARAMETERS = {
    "dataset_path": str(dataset_path),
    "queries": "10,12,18,21",
    "iterations": 1,
    "backends": ["combined_triton"],
    "results_dir": str(results_dir),
    "suffix": ".parquet",
    "scale_factor": scale_factor,
    "triton_executor": triton_executor,
    "exclude_conversion_time": True,
}


In [112]:
def display_parameters(parameters=PARAMETERS):
    """Display the active benchmark configuration."""
    return pd.Series(parameters, name="value").rename_axis("parameter").reset_index()


def benchmark_command(parameters=PARAMETERS, query=None):
    """Build the benchmark command."""
    command = [
        "python", "-m",
        "cudf_polars.streaming.benchmarks.compare_backends",
        query or parameters.get("queries") or "all",
        "--path", parameters["dataset_path"],
        "--iterations", str(parameters["iterations"]),
        "--suffix", parameters["suffix"],
        "--backends", ",".join(parameters["backends"]),
        "--executor", "streaming",
        "--triton-executor", parameters["triton_executor"],
        "--warmup",
        "--output-dir", parameters["results_dir"],
    ]
    if parameters.get("exclude_conversion_time", False):
        command.append("--exclude-conversion-time")
    return command


def query_ids(parameters=PARAMETERS):
    """Expand the configured query selector into individual query IDs."""
    selector = str(parameters.get("queries") or "all")
    if selector == "all":
        return list(range(1, 23))
    query_ids = []
    for item in selector.split(","):
        bounds = [int(value) for value in item.split("-", 1)]
        query_ids.extend(range(bounds[0], bounds[-1] + 1))
    return query_ids


def run_benchmark(parameters=PARAMETERS):
    """Run queries independently, returning successful records and failed IDs."""
    dataset = Path(parameters["dataset_path"])
    if not ((dataset / "lineitem.parquet").exists() or (dataset / "lineitem").is_dir()):
        raise FileNotFoundError(f"PDS-H dataset was not found at {dataset}")
    records = []
    failed_queries = []
    for query in query_ids(parameters):
        completed = subprocess.run(
            benchmark_command(parameters, str(query)),
            cwd=repo_root,
            env={**os.environ, "PYTHONPATH": str(repo_root / "python" / "cudf_polars")},
            check=False,
            text=True,
            capture_output=True,
        )
        if completed.stdout:
            print(completed.stdout)
        summary_path = Path(parameters["results_dir"]) / "summary.json"
        if completed.returncode != 0 or not summary_path.exists():
            failed_queries.append(query)
            print(f"Query {query} failed; continuing with the remaining queries.")
            if completed.stderr:
                print(completed.stderr)
            continue
        records.extend(load_summary(summary_path, parameters))
    return records, failed_queries


def load_summary(path, parameters=PARAMETERS):
    """Load summary.json as records keyed by the simulation dimensions."""
    summary = json.loads(Path(path).read_text())
    records = []
    for backend, queries in summary["timings_seconds"].items():
        for query, timing in queries.items():
            records.append({
                "query": int(query),
                "scale_factor": parameters["scale_factor"],
                "backend": backend,
                "exclude_conversion_time": parameters["exclude_conversion_time"],
                "mean_seconds": timing["mean"],
                "samples": timing["samples"],
                "speedup_vs_cpu": summary.get("speedup_vs_cpu", {}).get(backend, {}).get(query),
            })
    return records


def simulation_path(record, root=analysis_results_dir):
    """Return the analysis file for one query, scale, backend, and timing mode."""
    suffix = "_exclude_conversion_time" if record["exclude_conversion_time"] else ""
    return (
        root
        / f"q{record['query']}"
        / f"{record['scale_factor']:g}"
        / f"{record['backend']}{suffix}"
    )


def save_simulation(records, root=analysis_results_dir):
    """Append each new duration to its analysis result file."""
    for record in records:
        path = simulation_path(record, root)
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("a") as output:
            output.write(f"{record['mean_seconds']}\n")


def load_simulations(root=analysis_results_dir):
    """Load all saved durations from the analysis result files."""
    if not root.exists():
        raise FileNotFoundError(f"Analysis results folder was not found at {root}")
    records = []
    for path in root.glob("q*/*/*"):
        if not path.is_file():
            continue
        query = int(path.parent.parent.name.removeprefix("q"))
        scale_factor = float(path.parent.name)
        exclude_conversion_time = path.name.endswith("_exclude_conversion_time")
        backend = path.name.removesuffix("_exclude_conversion_time")
        records.extend(
            {
                "query": query,
                "scale_factor": scale_factor,
                "backend": backend,
                "exclude_conversion_time": exclude_conversion_time,
                "mean_seconds": float(duration),
            }
            for duration in path.read_text().splitlines()
            if duration
        )
    return pd.DataFrame(records)


In [113]:
display_parameters()
print("Running:", "PYTHONPATH=python/cudf_polars " + " ".join(shlex.quote(part) for part in benchmark_command()))
simulation_records, failed_queries = run_benchmark()
save_simulation(simulation_records)
print(f"Failed queries: {failed_queries}")
pd.DataFrame(simulation_records).drop(columns="samples")


Running: PYTHONPATH=python/cudf_polars python -m cudf_polars.streaming.benchmarks.compare_backends 10,12,18,21 --path /home/andri/polars-benchmark/data/tables/scale-5.0 --iterations 1 --suffix .parquet --backends combined_triton --executor streaming --triton-executor in-memory --warmup --output-dir /home/andri/cudf/tpch_backend_results/scale-5 --exclude-conversion-time
Query 10 - Iteration 0 finished in 0.3892s
Query 10 - Iteration 1 finished in 0.2878s
✅ Query 10 - Iteration 0 finished in 0.7896s
✅ Query 10 - Iteration 1 finished in 0.5057s

Validation Summary
✅ All validated queries passed.
cpu q10: 0.287751s
combined_triton q10: 0.505675s, speedup_vs_cpu=0.569x
Wrote comparison to /home/andri/cudf/tpch_backend_results/scale-5/summary.json

Query 12 - Iteration 0 finished in 0.2444s
Query 12 - Iteration 1 finished in 0.1421s
✅ Query 12 - Iteration 0 finished in 1.0381s
✅ Query 12 - Iteration 1 finished in 0.2718s

Validation Summary
✅ All validated queries passed.
cpu q12: 0.142117s


,query,scale_factor,backend,exclude_conversion_time,mean_seconds,speedup_vs_cpu
0,10,5,cpu,True,0.287751,NaN
1,10,5,combined_triton,True,0.505675,0.569043
2,12,5,cpu,True,0.142117,NaN
3,12,5,combined_triton,True,0.271789,0.522895
4,18,5,cpu,True,0.742146,NaN
5,18,5,combined_triton,True,0.326184,2.275235
6,21,5,cpu,True,1.404552,NaN
7,21,5,combined_triton,True,0.575606,2.440128
